# Transfer learning and imaging foundation models: worked solution

Worked solution for the exercise in the lesson **[Transfer learning and imaging foundation models](https://ramkumarmd.com/medical-ai/deep-learning-imaging/transfer-learning-imaging-foundation-models)** (*Deep learning & imaging* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](05-transfer-learning-imaging-foundation-models.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Transfer learning reuses a model trained on a large dataset as the starting point for a new task, so far fewer labelled images are needed. You can fine-tune all of its layers on the new data or freeze the learned features and retrain only the final layer. Imaging foundation models take this further: pretrained on millions of unlabelled medical images, such as retinal photographs, they can be adapted to many tasks with little labelled data. A pretrained model still has to be validated at each new site, because used as it is, it can fail on a different scanner.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q medmnist

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[CNNs for chest X-ray and retinal images](https://ramkumarmd.com/medical-ai/deep-learning-imaging/cnns-medical-imaging)**: Synthetic scans, a two-layer CNN and its test accuracy. Takes a few seconds on a CPU.

In [ ]:
import numpy as np
import torch
from torch import nn

def make_images(n, seed, marker_with_lesion=0.0):
    """Synthetic 28x28 'scans': noisy background; half contain a faint round lesion at a random place.
    marker_with_lesion: chance that lesion images also carry a bright corner marker (a shortcut)."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:28, :28]
    images = rng.normal(0, 1, (n, 28, 28))
    labels = rng.integers(0, 2, n)
    for i in np.where(labels == 1)[0]:
        cy, cx, r = rng.integers(6, 22), rng.integers(6, 22), rng.uniform(2.5, 4.0)
        images[i] += 1.0 * ((yy - cy) ** 2 + (xx - cx) ** 2 < r ** 2)
        if rng.random() < marker_with_lesion:
            images[i, :3, :3] += 4.0                     # e.g. a portable-scanner label
    return torch.tensor(images, dtype=torch.float32).unsqueeze(1), torch.tensor(labels)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # 28 -> 14
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 14 -> 7
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 2))
    def forward(self, x):
        return self.head(self.features(x))

def train(model, X, y, epochs=8, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        for idx in torch.randperm(len(X)).split(64):
            opt.zero_grad()
            nn.functional.cross_entropy(model(X[idx]), y[idx]).backward()
            opt.step()
    return model

def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

X_train, y_train = make_images(4000, seed=1)
X_test, y_test = make_images(1000, seed=2)
torch.manual_seed(0)
cnn = train(SmallCNN(), X_train, y_train)
print(f"CNN test accuracy: {accuracy(cnn, X_test, y_test):.3f}")
print(f"Parameters: {sum(p.numel() for p in cnn.parameters()):,}")

# CNN test accuracy: 0.946
# Parameters: 2,818

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Transfer to a new scanner with 100 images

Run the first block of the CNN lesson first. Compare four ways of getting a model for site B.

In [ ]:
import copy

def new_scanner(n, seed):
    """Target site: a different scanner gives lower contrast and an intensity offset."""
    X, y = make_images(n, seed)
    return 0.6 * X + 0.5, y

X_source, y_source = make_images(4000, seed=10)          # plenty of labelled images from site A
X_small, y_small = new_scanner(100, seed=11)              # only 100 labelled images at site B
X_target_test, y_target_test = new_scanner(1000, seed=12)

torch.manual_seed(0)
pretrained = train(SmallCNN(), X_source, y_source)
print(f"Pretrained model, applied to site B as is:  {accuracy(pretrained, X_target_test, y_target_test):.3f}")

torch.manual_seed(0)
scratch = train(SmallCNN(), X_small, y_small, epochs=30)
print(f"Trained from scratch on 100 site-B images:  {accuracy(scratch, X_target_test, y_target_test):.3f}")

fine_tuned = train(copy.deepcopy(pretrained), X_small, y_small, epochs=30)
print(f"Pretrained, fine-tuned on 100 site-B images: {accuracy(fine_tuned, X_target_test, y_target_test):.3f}")

# Pretrained model, applied to site B as is:  0.504
# Trained from scratch on 100 site-B images:  0.724
# Pretrained, fine-tuned on 100 site-B images: 0.854

### Freezing the learned features

Continuing from the previous block: keep the convolutional filters fixed and retrain only the final layer.

In [ ]:
frozen = copy.deepcopy(pretrained)
for p in frozen.features.parameters():
    p.requires_grad = False                       # keep the learned filters; retrain only the head
frozen.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 2))
torch.manual_seed(0)
opt = torch.optim.Adam(frozen.head.parameters(), lr=1e-3)
for _ in range(30):
    for idx in torch.randperm(len(X_small)).split(64):
        opt.zero_grad()
        nn.functional.cross_entropy(frozen(X_small[idx]), y_small[idx]).backward()
        opt.step()
print(f"Frozen features, new head on 100 images:    {accuracy(frozen, X_target_test, y_target_test):.3f}")
trainable = sum(p.numel() for p in frozen.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable} of {sum(p.numel() for p in frozen.parameters())}")

# Frozen features, new head on 100 images:    0.772
# Trainable parameters: 1570 of 2818

## Exercise

**Goal:** Compare transfer-learning strategies on synthetic and real images, and plan the validation of a pretrained model.

**Data:** Synthetic scans in the notebook; optional MedMNIST RetinaMNIST or PneumoniaMNIST; optional torchvision pretrained ResNet-18 (Generated in the notebook; MedMNIST under CC BY 4.0; check the licence of any pretrained weights)

### Step 1

Reproduce the four strategies and repeat with 25, 100 and 500 site-B images; plot accuracy against the number of labelled images.

In [ ]:
import matplotlib.pyplot as plt

def freeze_and_retrain_head(base, X, y, epochs=30):
    m = copy.deepcopy(base)
    for p in m.features.parameters():
        p.requires_grad = False
    m.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 2))
    torch.manual_seed(0)
    opt = torch.optim.Adam(m.head.parameters(), lr=1e-3)
    for _ in range(epochs):
        for idx in torch.randperm(len(X)).split(64):
            opt.zero_grad(); nn.functional.cross_entropy(m(X[idx]), y[idx]).backward(); opt.step()
    return m

as_is = accuracy(pretrained, X_target_test, y_target_test)
rows = []
for n_b in [25, 100, 500]:
    Xb, yb = new_scanner(n_b, seed=20 + n_b)
    torch.manual_seed(0)
    rows.append({"site-B images": n_b, "applied as is": as_is,
                 "from scratch": accuracy(train(SmallCNN(), Xb, yb, epochs=30), X_target_test, y_target_test),
                 "frozen features": accuracy(freeze_and_retrain_head(pretrained, Xb, yb), X_target_test, y_target_test),
                 "fine-tuned": accuracy(train(copy.deepcopy(pretrained), Xb, yb, epochs=30), X_target_test, y_target_test)})
import pandas as pd
curve = pd.DataFrame(rows).set_index("site-B images")
print(curve.round(3).to_string())

fig, ax = plt.subplots(figsize=(5.5, 3.5))
for col in curve.columns:
    ax.plot(curve.index, curve[col], "o-", label=col)
ax.set(xscale="log", xlabel="Labelled site-B images", ylabel="Site-B accuracy", xticks=curve.index)
ax.set_xticklabels(curve.index); ax.legend(fontsize=8)
plt.show()

With few labelled images, starting from the pretrained model helps most: at 25 site-B images, training from scratch
is no better than chance (0.511) while frozen features reach 0.672. As labelled site-B data grow, the gap narrows
(fine-tuned minus scratch: 0.18 at 100 images, 0.10 at 500) but has not closed. That is the general pattern behind foundation models: they matter most where labels are
scarce, which in medicine is most of the time.

### Step 2

Normalise each site's images (subtract the mean, divide by the standard deviation) and see whether the pretrained model works at site B without fine-tuning.

In [ ]:
def standardise(X):
    """Per-site normalisation: subtract that site's mean and divide by its standard deviation."""
    return (X - X.mean()) / X.std()

pretrained_std = train(SmallCNN(), standardise(X_source), y_source)
print(f"Pretrained on raw site-A images, site B as is:          {as_is:.3f}")
print(f"Pretrained on standardised site A, standardised site B: "
      f"{accuracy(pretrained_std, standardise(X_target_test), y_target_test):.3f}")

The site-B shift here is purely a contrast change and an intensity offset, so standardising each site's images
removes it, and the pretrained model works at site B with no fine-tuning at all. Real scanner differences (noise
texture, resolution, artefacts, different patient mix) are rarely this simple, but intensity normalisation is a
cheap first step that should always be tried, and checked, before fine-tuning.

### Step 3

Optional: fine-tune an ImageNet-pretrained ResNet-18 on RetinaMNIST and compare with training the same network from scratch.

In [ ]:
# Optional: an ImageNet-pretrained ResNet-18 (torchvision) on RetinaMNIST, compared with the same network from scratch.
# Downloads about 45 MB of weights and 3 MB of images; a few minutes on a CPU.
from medmnist import RetinaMNIST
from torchvision.models import ResNet18_Weights, resnet18

def load_retina(split):
    d = RetinaMNIST(split=split, download=True, root="data", size=28)
    X = torch.tensor(d.imgs, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    X = nn.functional.interpolate(X, size=64, mode="bilinear", align_corners=False)   # small images, upsampled
    mean, std = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return (X - mean) / std, torch.tensor(d.labels.ravel(), dtype=torch.long)

Xr_tr, yr_tr = load_retina("train")
Xr_te, yr_te = load_retina("test")

def resnet(pretrained_weights):
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained_weights else None)
    m.fc = nn.Linear(m.fc.in_features, 5)                    # five retinopathy grades (0-4)
    return m

def fit_eval(m, epochs=6, lr=1e-3):
    torch.manual_seed(0)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    m.train()
    for _ in range(epochs):
        for idx in torch.randperm(len(Xr_tr)).split(64):
            opt.zero_grad(); nn.functional.cross_entropy(m(Xr_tr[idx]), yr_tr[idx]).backward(); opt.step()
    m.eval()
    with torch.no_grad():
        pred = m(Xr_te).argmax(1)
    exact = (pred == yr_te).float().mean().item()
    within1 = ((pred - yr_te).abs() <= 1).float().mean().item()
    return exact, within1

for label, weights in [("ResNet-18 from scratch", False), ("ResNet-18, ImageNet-pretrained", True)]:
    exact, within1 = fit_eval(resnet(weights))
    print(f"{label:<32} exact grade {exact:.3f}   within one grade {within1:.3f}")

RetinaMNIST has only 1,080 training images, graded 0-4 (MedMNIST, CC BY 4.0), so it is hard at 28x28 and accuracy
is modest for both networks. Grading is ordinal, which is why "within one grade" is reported too. In our run the
ImageNet-pretrained network was slightly better on the exact grade (0.527 against 0.498) and no better within one
grade (0.765 against 0.770): a small, uncertain gain on 400 test images. Results vary with seeds and epochs on a
dataset this small, so compare on your own run rather than assuming pretraining helps.

### Step 4

Write the validation plan you would require before using an imaging foundation model in a diabetic retinopathy screening programme.

Example validation plan before using an imaging foundation model in a diabetic retinopathy screening programme:

1. **Intended use** written down: the grading task (referable retinopathy and maculopathy, or full grading), the
   population, the cameras, and whether the model replaces, assists or quality-checks human graders.
2. **Regulatory status** for that intended use, and the manufacturer's evidence: what data the foundation model
   and the final classifier were trained on, and whether any of our own images were included.
3. **Local retrospective validation** on a large, consecutive sample from our programme (several thousand images,
   with enough referable cases), graded by certified graders with arbitration as the reference standard:
   sensitivity for referable disease (the priority), specificity, ungradable rate, with confidence intervals.
4. **Subgroups**: each camera model and site, mydriatic versus non-mydriatic images, age, sex, ethnicity, and image
   quality; acceptance criteria for each, set in advance.
5. **Comparison with current practice**: the programme's existing grader performance on the same images.
6. **Prospective silent run**: the model grades alongside the usual pathway for a few months, without affecting care.
7. **Failure handling**: ungradable and borderline images always go to a human grader; no image is discarded unseen.
8. **Monitoring after go-live**: referral rates, ungradable rates and a graded random sample each month; re-validation
   after any model update or new camera; an owner and a switch-off rule.

---

Lesson: [https://ramkumarmd.com/medical-ai/deep-learning-imaging/transfer-learning-imaging-foundation-models](https://ramkumarmd.com/medical-ai/deep-learning-imaging/transfer-learning-imaging-foundation-models). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.